#  如果 PPO 训练后模型出现"模式崩溃"（只输出固定句式），可能的原因？

PPO训练后模型出现“模式崩溃”（只输出固定句式）的根本原因是:
- <font color='red'>策略在强化学习中**找到了**奖励函数的“捷径”**并锁死在局部最优**</font>
  - 核心解决路径是:<font color='red'>增大KL惩罚系数、引入熵正则化、降低学习率、减少训练步数</font>。

## 现象与诊断
模型无论输入什么，都反复输出同一句话或固定模板（如“是的，是的，是的…”或“综上所述……”）。训练日志中通常伴随以下异常信号：
- Reward 骤增：
  - <font color='red'>策略找到“捷径”欺骗奖励模型获取虚高分数</font>
  
- KL 散度骤增：
  - 策略分布剧烈偏移，完全偏离参考模型。
  
- Perplexity(困惑度) 骤降：
  - 模型对生成结果“过度确定”，<font color='red'>已拟合到固定模式</font>
  
- 策略熵骤降：
  - 正常LLM约 8-12 bits，若连续低于 5 bits 说明熵坍塌。

- Response Length 骤增或骤缩：
  - 输出长度异常变化。

## 根本原因
- 奖励黑客（Reward Hacking）：
  - 策略空间极大，<font color='red'>模型发现某种固定输出模式能稳定获得高分，便“锁定”该模式，忽略实际语义质量。</font>
- KL 惩罚系数 β 过小：
  - β < 0.01 时约束几乎失效，<font color='red'>策略可自由偏离参考模型</font>，迅速收窄到单一输出分布。
  
- 熵坍塌（Entropy Collapse）：
  - <font color='red'>策略熵过低导致输出多样性下降，陷入局部最优</font>
  
- Critic 过估计：
  - 价值网络过高估计状态价值，导致优势估计为负，策略异常保守，倾向于输出“最安全”的固定回复。
  
- 学习率过大：
  - 单步梯度更新导致策略分布发生质变，直接跳入崩溃区域。
  
- 过度训练：
  - 模式崩溃常发生在训练后期，模型反复优化同一模式导致不可逆退化。

## 解决方案（按优先级排序）

###  增大 KL 惩罚系数 β（最关键）
<font color='red'>这是最常见且最有效的手段</font>。β 太小导致模型“放飞自我”追逐高分，增大后可强制约束策略不偏离参考模型太远。

- 将 β 从 0.01 调至 0.05 ~ 0.1，可显著稳定训练。
- <font color='red'>推荐使用自适应 KL 控制</font>：
  - KL > target_KL × 1.5 时增大 β，
  - KL < target_KL / 1.5 时减小 β。

###  加入熵正则化（Entropy Bonus）

在 PPO 目标函数中加入熵奖励项，<font color='red'>鼓励策略保持多样性</font>，防止过早收敛到单一模式。
- 推荐 entropy_coef 设为 0.01~0.1。
- <font color='red'>当策略熵连续 5 步低于阈值时，可动态提升熵系数（×2），熵恢复后调回。</font>

###  降低学习率
- <font color='red'>将学习率降至原来的 1/5~1/10，从崩溃前的稳定 checkpoint 继续训练</font>
- <font color='red'>训练早期建议使用极保守的学习率（如 1e-7，比 SFT 低 10 倍），稳定后再逐步调整</font>

###  减少训练步数
- 模式崩溃通常<font color='red'>发生在过度训练后</font>。
  - 若 reward 曲线已出现“早峰-断崖”形态，最佳策略往往在崩溃前就已出现，应提前保存 checkpoint 并回滚。

###  奖励归一化与裁剪
- 对同一批次奖励做 Z-score 归一化（均值 0，标准差 1），<font color='red'>消除批次间尺度差异</font>
- 在 [-5, +5] 范围内裁剪归一化后的奖励，<font color='red'>防止极端样本主导梯度</font>

###  Checkpoint 监控与回滚
   - <font color='red'>每 50~100 步保存一次 checkpoint</font>
   - <font color='blue'>若 reward 骤增但 KL 同步骤增，说明可能已崩溃，应立即回滚到之前的稳定 checkpoint。</font>
   
## 急救流程
若训练已崩溃，可按以下步骤尝试恢复：
1. 回滚到崩溃前最后一个稳定的 checkpoint。
2. 降低学习率至原来的 1/5~1/10。
3. 增大 KL 惩罚系数 β（如从 0.01 调至 0.05）。
4. 加入/增大熵正则化系数。
5. 从小学习率 warm-up 几百步后，继续训练并密切监控 KL、熵、reward 三个指标。

若多次尝试仍无法恢复，建议检查奖励模型质量（是否存在标注噪声或漏洞）或考虑换用 GRPO/DPO 等替代方案。